# Dataset Preparation
Six identities were selected from the finalized class identity pool: three male and three female identities. Each identity has 25 images, providing balanced class counts.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Start Jupyter in ie7615-project1 or ie7615-project1/notebooks")
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display
from src.config import DATA_DIR, SELECTED_IDS
from src.data_utils import prepare_pipeline_subset, audit_subset
from src.experiment import start_experiment

In [2]:
# Three male and three female identities selected from the finalized class pool.
RAW_DIR = ROOT / "data" / "raw"
print("Selected identities:", SELECTED_IDS)

Selected identities: [2619, 797, 2970, 4422, 3, 2336]


## Split Construction and Integrity Checks
Verify image counts, class consistency and image readability. Compare decoded-pixel hashes to identify exact duplicates across splits.

In [3]:
# Load the prepared split or create it using the configured identities and seed.
if not DATA_DIR.exists():
    prepare_pipeline_subset(RAW_DIR, DATA_DIR, SELECTED_IDS)
manifest, class_names = audit_subset(DATA_DIR)
if set(class_names) != set(map(str, SELECTED_IDS)):
    raise ValueError("Existing split does not match SELECTED_IDS")
display(manifest.groupby(["identity", "split"]).size().unstack()[["train", "val", "test"]])
display(pd.DataFrame({"class_index": range(len(class_names)), "CelebA_ID": class_names}))

split,train,val,test
identity,,,
2336,17,4,4
2619,17,4,4
2970,17,4,4
3,17,4,4
4422,17,4,4
797,17,4,4


,class_index,CelebA_ID
0,0,2336
1,1,2619
2,2,2970
3,3,3
4,4,4422
5,5,797


In [4]:
# Save the split manifest and identity-selection metadata.
import json
manifest.to_csv(DATA_DIR / "split_manifest.csv", index=False)
selection = {
    "selected_ids": SELECTED_IDS,
    "selection_source": "finalized class identity pool",
    "selection_criteria": "three male and three female identities; balanced image counts",
    "male_identity_count": 3,
    "female_identity_count": 3
}
(DATA_DIR / "selection_metadata.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
print("Manifest:", DATA_DIR / "split_manifest.csv")

Manifest: /home/zhang.jiaso/ondemand/data/sys/dashboard/batch_connect/sys/jupyterlab-ml-courses/ie7615-project1/data/selected_subset/split_manifest.csv


## Experiment Initialization
Create one experiment directory shared by the three training notebooks. This records the data manifest and executable sources before training.

In [5]:
RUN_DIR = start_experiment()
print("Experiment:", RUN_DIR)

Experiment: /home/zhang.jiaso/ondemand/data/sys/dashboard/batch_connect/sys/jupyterlab-ml-courses/ie7615-project1/runs/20260926T151014Z_08ffd273
